# Partition uncertainties in projections of impacts

For different sets of runs: 
- a "balanced panel" of GCMs / runs
- all available runs at GWLX
- with one reanalysis left out

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import copy
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from matplotlib import colors as mcolors

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle

dir_list = get_params()

In [ ]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

## Climate uncertainty partitioning

In [6]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':True},
               'mortality-interacted':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':True},
               'gdppc':{'varname':'tasmeanpoly','geo_typ':'adm1','popwght':True},
              'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

#params_proc = {'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}


In [ ]:
overwrite = True
for res in ['025deg','1deg']:
    for proc in params_proc:
        if res == '025deg':
            output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}.zarr'
        else:
            output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}_1deg.zarr'
        if overwrite or (not os.path.exists(output_fn)):
            #---------------------------
            # Setup
            #---------------------------
            # Load impacts projections
            if res == '025deg':
                fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}_agg_*ALLGWL*{'*popwght*'*params_proc[proc]['popwght']}.zarr'))
            elif res == '1deg':
                fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}_agg-1deg_*ALLGWL*{'*popwght*'*params_proc[proc]['popwght']}.zarr'))
            if len(fns) == 0:
                print(f'No files found for {res} {proc} data, skipping.')
                continue
            
            def load_impact(fn):
                ds_tmp = xr.open_zarr(fn,consolidated=False)
                ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
                ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})
            
                ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
                if 'idv' in ds_tmp.coords:
                    ds_tmp = ds_tmp.reset_index('idv')
                ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
                ds_tmp = ds_tmp.drop_vars(['flag_has_historical_duplicates','flag_resolution','flag_has_data','variable'],errors='ignore')
                return ds_tmp
            
            with warnings.catch_warnings():
                # Silence zarr warnings related to the /.done flags used
                warnings.filterwarnings('ignore')
                dss = xr.concat([load_impact(fn) for fn in fns],
                            dim='idv',join='outer')
            # For interacted, the signal is the difference between 
            # "dmortality" (assuming climate + society changing) and
            # "dmortality_ref" (assuming climate constant, society changing)
            if proc == 'mortality-interacted':
                dss['dmortality-interacted'] = dss['dmortality'] - dss['dmortality_ref']
                dss = dss.drop_vars(['dmortality','dmortality_ref'])
            
            dss = dss.chunk({'poly_idx':50,'idv':-1,'proj_base':-1,'gwl':-1})
    
            # Save temporary rechunked file to make dask graphing more stable
            tmp_fn = dir_list['proc']+'tmp/uncert_rechunk_cache_tmp.zarr'
            dss = dss.reset_index('idv')
            for v in dss.coords:
                dss[v] = dss[v].load()
            utility_save(dss.drop_encoding(),tmp_fn,zarr_mode = 'w')
            dss = xr.open_zarr(tmp_fn)
            for v in ['model','exprun','experiment','run']:
                # Explicitly cast these dims as objects because of a numpy bug
                dss[v] = dss[v].astype(object).load()
            dss = dss.set_index(idv = ['model','exprun']).drop_vars(['experiment','run'])
    
            # Figure out how many BCD-ME members reach a given GWL
            # (copy.deepcopy() necessary as a workaround around a numpy bug
            # affecting the unstack a few lines down that's triggered 
            # indirectly by the reset_index here)
            gwl_counter = copy.deepcopy(dss.maxgwl).load().reset_index('idv').groupby(['model','maxgwl']).count()
            gwl_counter = gwl_counter.sel(maxgwl = gwl_counter.maxgwl.values[::-1]).cumsum('maxgwl')
            #print(gwl_counter)
            
            uncerts_all = []
            #---------------------------
            # Uncertainty partitioning over all runs
            # (for all LEs, defined as >= 5 runs)
            #---------------------------
            nruns = 5
    
            gwlmods = gwl_counter.sel(maxgwl=1.5)
            proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
            
            dss_tmp = dss.isel(idv=[m in proc_mods for m in dss.model.values]).chunk({'idv':-1})
            dss_tmp = dss_tmp.unstack('idv')['d'+proc]
            dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
            
            # Calculate uncertainty partitioning
            with warnings.catch_warnings():
                # Ignoring chunk increase warnings and divide by 0s in uncertainty 
                # partitioning (happens over ocean, e.g.)
                warnings.filterwarnings('ignore')
                uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

            uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
            uncerts = uncerts.expand_dims({'sample':['allruns']})
            
            uncerts_all.append(uncerts)
            
            #---------------------------
            # Uncertainty partitioning over a balanced panel up to GWL 3
            #---------------------------
            max_gwl = 3
            if proc == 'payroll':
                nruns = 10
            else:
                nruns = 13
            
            gwlmods = gwl_counter.sel(maxgwl=3)
            proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
            
            dss_tmp = dss.isel(idv = dss.maxgwl.load()>=max_gwl).groupby('model').map(lambda x: x.isel(idv=slice(0, nruns)))
            dss_tmp = dss_tmp.isel(idv=[m in proc_mods for m in dss_tmp.model.values]).chunk({'idv':-1})
            dss_tmp = dss_tmp.unstack('idv')['d'+proc]
            dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
            
            with warnings.catch_warnings():
                # Ignoring chunk increase warnings and divide by 0s in uncertainty 
                # partitioning (happens over ocean, e.g.)
                warnings.filterwarnings('ignore')
                uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

            uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
            uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods']})
            
            uncerts_all.append(uncerts)
    
            #---------------------------
            # Uncertainty partitioning over a balanced panel up to GWL 3
            # withholding a reanalysis each time
            #---------------------------
            for proj_base in dss.proj_base.values:
                max_gwl = 3
                if proc == 'payroll':
                    nruns = 10
                else:
                    nruns = 13
                
                gwlmods = gwl_counter.sel(maxgwl=3)
                proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
                
                dss_tmp = dss.isel(idv = dss.maxgwl.load()>=max_gwl).groupby('model').map(lambda x: x.isel(idv=slice(0, nruns)))
                dss_tmp = dss_tmp.sel(proj_base = [pb for pb in dss_tmp.proj_base.values if pb != proj_base])
                dss_tmp = dss_tmp.isel(idv=[m in proc_mods for m in dss_tmp.model.values]).chunk({'idv':-1})
                dss_tmp = dss_tmp.unstack('idv')['d'+proc]
                dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
                
                with warnings.catch_warnings():
                    # Ignoring chunk increase warnings and divide by 0s in uncertainty 
                    # partitioning (happens over ocean, e.g.)
                    warnings.filterwarnings('ignore')
                    uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

                uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
                uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods_no{proj_base}']})
                
                uncerts_all.append(uncerts)
            
            #---------------------------
            # Export
            #---------------------------
            # Concatenate into single ds
            uncerts_all = xr.concat(uncerts_all,dim='sample')
            uncerts_all = uncerts_all.set_coords('numruns')
            uncerts_all = uncerts_all.where(uncerts_all.sum('typ')>0)
            
            # Export
            uncerts_all.attrs = {'SOURCE':'4_partition_uncertainty.ipynb',
                                 'DESCRIPTION':f'Uncertainty partitioning of projections of {proc} between model, internal, and reanalysis sources'}

            with warnings.catch_warnings():
                # Ignoring chunk increase warnings and divide by 0s in uncertainty 
                # partitioning (happens over ocean, e.g.)
                warnings.filterwarnings('ignore')
                utility_save(uncerts_all,output_fn,zarr_mode = 'w')
    
            os.system('rm -rf '+tmp_fn)
            print('Temporary file '+tmp_fn+' removed.')
        else:
            print(output_fn+' exists!')
    
